In [0]:
%run ./01_setup_environment

In [0]:
# ========================================
# 07_incremental_merge_cdc
# ========================================

from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.window import Window

try:

    bronze_patients_df = spark.read.format("delta") \
        .load(f"{bronze_path}/validated_patients")

    # ----------------------------------------
    # Keep Latest Record Per Patient
    # ----------------------------------------

    window_spec = Window.partitionBy(
        "patient_id"
    ).orderBy(
        col("ingestion_time").desc()
    )

    latest_patients_df = bronze_patients_df.withColumn(
        "row_num",
        row_number().over(window_spec)
    ).filter(
        col("row_num") == 1
    ).drop("row_num")

    # ----------------------------------------
    # Delta Table Reference
    # ----------------------------------------

    silver_table = DeltaTable.forPath(
        spark,
        f"{silver_path}/patients_clean"
    )

    # ----------------------------------------
    # CDC Merge
    # ----------------------------------------

    silver_table.alias("target").merge(
        source=latest_patients_df.alias("source"),
        condition="target.patient_id = source.patient_id"
    ).whenMatchedUpdate(set={
        "patient_name": "source.patient_name",
        "city": "source.city",
        "gender": "source.gender",
        "updated_at": "current_timestamp()"
    }).whenNotMatchedInsert(values={
        "patient_id": "source.patient_id",
        "patient_name": "source.patient_name",
        "city": "source.city",
        "gender": "source.gender",
        "updated_at": "current_timestamp()"
    }).execute()

    # ----------------------------------------
    # Audit Logging
    # ----------------------------------------

    log_audit(
        "patients_cdc_pipeline",
        "silver",
        "patients_clean",
        latest_patients_df.count(),
        "SUCCESS"
    )

    print("CDC Merge Completed Successfully")

except Exception as e:

    log_audit(
        "patients_cdc_pipeline",
        "silver",
        "patients_clean",
        0,
        "FAILED",
        str(e)
    )

    raise e